# PRC Data Challenge 2025: reproduce submission v6 and its corrected validation

Run in Google Colab (*Runtime → Run all*). The notebook:

1. downloads the challenge data from Zenodo (record 19184662, CC BY 4.0, about 4 GB for the train and ranking trajectories);
2. builds the segment features (cached in your Drive, since the training set takes a while);
3. trains the v6 model with the **original** procedure and reports its validation RMSE;
4. predicts the ranking intervals and scores them against the published ground truth (`fuel_rank.parquet`);
5. runs the **corrected** flight-grouped cross-validation;
6. optionally rescores your past submission files.

The ANP aircraft files (`Aircraft.csv`, `Aerodynamic_coefficients.csv`) are not redistributed: download them from the
[EASA ANP site](https://www.easa.europa.eu/en/domains/environment/policy-support-and-research/aircraft-noise-and-performance-anp-data)
and put them in `ANP_DIR` below.

**Do not put any access key in this notebook.** The challenge bucket credentials are not needed here.

In [ ]:
!pip -q install git+https://github.com/MoustaphaRouf/prc2025-fuel-burn

In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

WORK = Path('/content/drive/MyDrive/MES_RECHERCHES/PRC_2025/repro')   # feature cache and outputs
ANP_DIR = Path('/content/drive/MyDrive/MES_RECHERCHES/PRC_2025')       # Aircraft.csv, Aerodynamic_coefficients.csv
SUBMISSIONS = Path('/content/drive/MyDrive/MES_RECHERCHES/PRC_2025')    # honorable-yogurt_v*.parquet (optional)
DATA = Path('/content/data')                                            # local disk: faster than Drive for 4 GB
WORK.mkdir(parents=True, exist_ok=True)

In [ ]:
from prcfuel import data
files = data.download(['fuel_train', 'fuel_rank', 'flightlist_train', 'flightlist_rank',
                       'flights_train', 'flights_rank'], DATA)
fuel_train = data.read_fuel(files['fuel_train'])
fuel_rank = data.read_fuel(files['fuel_rank'])            # ranking intervals with ground truth
fl_train = data.read_flight_list(files['flightlist_train'])
fl_rank = data.read_flight_list(files['flightlist_rank'])
print(len(fuel_train), 'training intervals,', len(fuel_rank), 'ranking intervals')

In [ ]:
import pandas as pd
from prcfuel import features

def cached(name, zip_key, fuel, flist):
    p = WORK / f'{name}.parquet'
    if p.exists():
        return pd.read_parquet(p)
    paths = data.extract(files[zip_key], DATA / zip_key)
    df = features.build_dataset(paths, fuel, flist)
    df.to_parquet(p)
    return df

train_feats = cached('train_features', 'flights_train', fuel_train, fl_train)
rank_feats = cached('rank_features', 'flights_rank', fuel_rank.drop(columns='fuel_kg'), fl_rank)
print(train_feats.shape, rank_feats.shape)
print(len(fuel_rank) - len(rank_feats), 'ranking intervals without trajectory points (fallback)')

In [ ]:
from prcfuel import aircraft
perfo = aircraft.performance_table(aircraft.load_anp(ANP_DIR / 'Aircraft.csv',
                                                     ANP_DIR / 'Aerodynamic_coefficients.csv'))
perfo.head()

## 1. Original v6 procedure

Random row-level 80/20 split, early stopping on the same 20 %, ranking features clipped with their own quantiles.
The notebook reported a validation RMSE of 177.2 kg; the official score of the v6 file was 263.8 kg.

In [ ]:
from prcfuel import model, score
m, val_rmse, pred = model.reproduce_v6(train_feats, rank_feats, perfo)
fb = model.fallback_predictions(fuel_rank.drop(columns='fuel_kg'), fuel_train, fl_rank, fl_train)
sub = model.submission(fuel_rank[['idx', 'flight_id', 'start', 'end']], pred, fb)
s = score.score(sub, fuel_rank)
print(f'validation RMSE (row split): {val_rmse:.1f} kg')
print(f'ranking RMSE: {s["rmse_kg"]:.1f} kg on {s["matched"]} intervals ({sub.attrs["n_fallback"]} from the fallback)')
sub.to_parquet(WORK / 'v6_reproduced.parquet')

## 2. Corrected validation

Folds grouped by flight, clipping and target encoding fitted on the training fold only, early stopping on an inner
split of the training fold. This estimate should be much closer to the ranking RMSE than the row-split one.

In [ ]:
cv = model.grouped_cv(train_feats, perfo, n_splits=5)
cv.to_csv(WORK / 'grouped_cv.csv', index=False)
print(cv.round(1).to_string(index=False))
print(f'grouped CV RMSE: {cv.rmse.mean():.1f} ± {cv.rmse.std():.1f} kg')

## 3. Rescore past submissions (optional)

In [ ]:
rows = [{'file': p.name, **score.score(pd.read_parquet(p), fuel_rank)}
        for p in sorted(SUBMISSIONS.glob('honorable-yogurt_v*.parquet'))]
pd.DataFrame(rows).round(1)

Share the printed numbers (not this executed notebook) to update the README.